In [1]:
%load_ext dotenv
%dotenv config.env

In [2]:
%load_ext autoreload 
%autoreload 2

In [3]:
import os
import numpy as np
from pathlib import Path
from ipyfilechooser import FileChooser
from matplotlib import pyplot as plt
import h5py
from ipywidgets import Button, Layout, interactive, fixed
import ipywidgets as widgets
from IPython.display import display, clear_output
from datetime import datetime


In [4]:
# Select h5 file

path = Path(os.getenv('SMPP_INPUT_DIR'))
fc = FileChooser(path)
display(fc)


FileChooser(path='/scratch/project_2009817/SMP_python_images', filename='', title='', show_hidden=False, selec…

In [18]:
# read file
h5file = h5py.File(fc.selected, 'r')

In [19]:
def viewer_widget(frame_n):

    frame_selected_w = widgets.IntSlider(
        value=0,
        min=0,
        max=frame_n,
        step=1,
        description='frame:',
        disabled=False,
        continuous_update=True,
        orientation='horizontal',
        readout=True,
        readout_format='d',
        layout=Layout(width='800px'),
    ) 

    return frame_selected_w

def h5_view(h5_file_imdata, frame):
    plt.imshow(h5_file_imdata[frame,:,:])

frame_w = viewer_widget(h5file['ImageData'].shape[0]-1)
interactive_plot = interactive(h5_view, 
                               frame=frame_w, 
                               h5_file_imdata=fixed(h5file['ImageData']),
                               )
output = interactive_plot.children[-1]
# output.layout.height = '500px'
display(interactive_plot)



# metadata
def print_h5_meta(h5file):
    led=h5file['MetaData'].attrs['LEDpower']
    lasers = [str((i, h5file['MetaData']['Lasers'][i].attrs['On'])) for i in (h5file['MetaData']['Lasers'])]
    start=h5file['MetaData'].attrs['StartTime']
    start_hr=str(datetime.fromtimestamp(start).time())
    print(f"{fc.selected.split('/')[-1]} \nshape: {h5file['ImageData'].shape} \nstart: {start_hr}\nled: {led}\nlasers: {lasers}")
    
    for i in h5file.keys():
        print(i)
        print(' '*4, h5file[i])
        if isinstance(h5file[i], h5py.Group):
            for k in list(h5file[i].keys()):
                obj = h5file[i][k]
                print(' '*8, obj)
                for attr_name, attr_value in obj.attrs.items():
                    if obj.name == "/MetaData/temperature":
                        print(' '*12, attr_name)  
                    else:
                        print(' '*12, attr_name, attr_value) 


    print('')
    try:
        t = h5file['MetaData']['temperature'].attrs['temperature'].decode('utf-8')
        np_t = np.array([float(value[1:]) for value in t.split('\na') if value.startswith('F')])
        t = str(tuple(np.percentile(np_t, [10, 50, 90]))) 
    except KeyError:
        t = 'no temperature data'
    print(f'temperature: {t}')
    
    print('events:')
    for event in h5file['Events']:
        print(event)
    return
    

interactive(children=(IntSlider(value=0, description='frame:', layout=Layout(width='800px'), max=1998), Output…

In [17]:
print_h5_meta(h5file)
h5file.close()


260924_EN_40gem_Marine_broth_JF552_L.h5 
shape: (1999, 512, 512) 
start: 13:07:57.491890
led: 0.0
lasers: ["('l405', np.uint8(0))", "('l488', np.uint8(1))", "('l561', np.uint8(0))", "('l640', np.uint8(0))"]
Events
     <HDF5 dataset "Events": shape (2,), type "|V296">
ImageData
     <HDF5 dataset "ImageData": shape (1999, 512, 512), type "<u2">
MetaData
     <HDF5 group "/MetaData" (11 members)>
         <HDF5 group "/MetaData/Camera" (0 members)>
             Binning b'(I1\nI1\ntp0\n.'
             CLASS b'GROUP'
             CycleTime 0.1004749983549118
             EMGain 300
             ElectronsPerCount 4.65
             IntegrationTime 0.10000000149011612
             Model b'DU897_BV'
             Name b'Andor IXon DV97'
             NoiseFactor 1.41
             ROI b'(I0\nI0\nI512\nI512\ntp0\n.'
             ROIHeight 512
             ROIOriginX 0
             ROIOriginY 0
             ROIWidth 512
             ReadNoise 71.6
             SerialNumber 14305
             Start

In [ ]:
# Add missing H5 metadata

print('''
Input format:

Chemical 'None'
Dye 'JFX554'
Media 'M9glyCAAT'
Notes 'SYTO9_RT'
Species 'Escherichia coli'
Strain 'JML_Ec_59'
User 'Your Name'
''')


h5file = h5py.File(fc.selected, 'a')
grp = h5file.create_group('/MetaData/Sample')
for i in ['Chemical',
          'Dye',
          'Media',
          'Notes',
          'Species',
          'Strain',
          'User',
          ]:
    grp.attrs[i] = input(i+':').encode('utf-8')

print('\nUpdated meta:')
print_h5_meta(h5file)
h5file.close()



In [ ]:
# Correct existing H5 metadata

with h5py.File(fc.selected, 'r+') as h5file:
    value = round(float(input('set attr value:')), 2)
    h5file['MetaData'].attrs['LEDpower'] = np.float64(value)

with h5py.File(fc.selected, 'r+') as h5file:
    print('\nUpdated meta:')
    print_h5_meta(h5file)



